# FAIR Quality Assurance Scorer
**SME Metadata Governance Framework** | Murtadha Oduntan | 2025/A/MIT/0033

This notebook computes FAIR compliance scores for all datasets in the governance catalogue.
Run each cell in order to generate the full governance quality report.

In [ ]:
import os, requests, json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

API_BASE = os.environ.get('API_BASE_URL', 'http://localhost:8000')
EMAIL    = 'admin@smegov.ng'
PASSWORD = 'Admin@SMEGov2026'

# ── Authenticate ──────────────────────────────────────────────
resp  = requests.post(f'{API_BASE}/auth/login',
                      json={'email': EMAIL, 'password': PASSWORD})
token = resp.json()['access_token']
hdrs  = {'Authorization': f'Bearer {token}'}
print('✓ Authenticated as:', resp.json()['email'])

In [ ]:
# ── Compute FAIR scores for all datasets ──────────────────────
datasets_resp = requests.get(f'{API_BASE}/v1/datasets?per_page=100', headers=hdrs)
datasets      = datasets_resp.json()['items']
print(f'Found {len(datasets)} datasets in catalogue')

scored = []
for ds in datasets:
    r = requests.post(f"{API_BASE}/v1/fair/{ds['id']}/score", headers=hdrs)
    if r.status_code == 200:
        result = r.json()
        result['title'] = ds['title']
        result['sensitivity'] = ds['sensitivity_class']
        scored.append(result)
        print(f"  ✓ {ds['identifier']:30s} FAIR={result['fair_score']:5.1f}  {result['grade']}")
    else:
        print(f"  ✗ {ds['identifier']} — {r.status_code}")

df = pd.DataFrame(scored)
print(f'\nScored {len(df)} datasets')

In [ ]:
# ── Traffic-light dashboard ───────────────────────────────────
def traffic_light(score):
    if score >= 75: return '#27AE60'   # green
    if score >= 50: return '#F39C12'   # amber
    return '#E74C3C'                   # red

if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('FAIR Quality Assurance Dashboard', fontsize=14, fontweight='bold', color='#1F3864')

    # Bar chart — FAIR score per dataset
    ax1 = axes[0]
    colors = [traffic_light(s) for s in df['fair_score']]
    bars = ax1.barh(df['title'].str[:30], df['fair_score'], color=colors, edgecolor='white')
    ax1.axvline(x=75, color='#27AE60', linestyle='--', lw=1.5, label='Target (75)')
    ax1.axvline(x=50, color='#F39C12', linestyle='--', lw=1.5, label='Minimum (50)')
    ax1.set_xlim(0, 100)
    ax1.set_xlabel('FAIR Score', fontsize=11)
    ax1.set_title('Dataset FAIR Scores', fontsize=11, fontweight='bold')
    ax1.legend(fontsize=9)
    for bar, score in zip(bars, df['fair_score']):
        ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{score:.0f}', va='center', fontsize=9)

    # Radar / grouped bar — average scores by principle
    ax2 = axes[1]
    principles = ['Findable\n(F)', 'Accessible\n(A)', 'Interoperable\n(I)', 'Reusable\n(R)']
    avgs = [df['f_score'].mean(), df['a_score'].mean(),
            df['i_score'].mean(), df['r_score'].mean()]
    bar_colors = ['#2E74B5','#1D6A72','#B8860B','#C0392B']
    ax2.bar(principles, avgs, color=bar_colors, edgecolor='white', width=0.5)
    ax2.axhline(y=20, color='#27AE60', linestyle='--', lw=1.5, label='Target (80%)')
    ax2.set_ylim(0, 25)
    ax2.set_ylabel('Average Score (max 25)', fontsize=11)
    ax2.set_title('Average Score by FAIR Principle', fontsize=11, fontweight='bold')
    ax2.legend(fontsize=9)
    for i, (v, c) in enumerate(zip(avgs, bar_colors)):
        ax2.text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=10, color=c, fontweight='bold')

    plt.tight_layout()
    plt.savefig('fair_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Dashboard saved as fair_dashboard.png')

In [ ]:
# ── Summary statistics ────────────────────────────────────────
if not df.empty:
    avg = df['fair_score'].mean()
    print('═' * 55)
    print(' FAIR QUALITY ASSURANCE — CATALOGUE SUMMARY')
    print('═' * 55)
    print(f'  Total datasets scored  : {len(df)}')
    print(f'  Average FAIR score     : {avg:.1f} / 100')
    print(f'  Highest score          : {df["fair_score"].max():.1f}')
    print(f'  Lowest score           : {df["fair_score"].min():.1f}')
    print(f'  Datasets ≥75 (Good)    : {(df["fair_score"]>=75).sum()}')
    print(f'  Datasets <50 (At Risk) : {(df["fair_score"]<50).sum()}')
    print()
    print(f'  Avg Findable           : {df["f_score"].mean():.1f} / 25')
    print(f'  Avg Accessible         : {df["a_score"].mean():.1f} / 25')
    print(f'  Avg Interoperable      : {df["i_score"].mean():.1f} / 25')
    print(f'  Avg Reusable           : {df["r_score"].mean():.1f} / 25')
    print('═' * 55)

In [ ]:
# ── Remediation priority list ─────────────────────────────────
summary_resp = requests.get(f'{API_BASE}/v1/fair/catalogue/summary', headers=hdrs)
if summary_resp.status_code == 200:
    summary = summary_resp.json()
    print(f'Overall Catalogue Grade: {summary["overall_grade"]}')
    print(f'Average FAIR Score: {summary["average_fair"]}')
    unscored = summary.get('unscored', 0)
    if unscored:
        print(f'\n⚠ {unscored} dataset(s) have no FAIR score — run scoring cell above first')